# Scoring variants on their splicing effect

This notebook demonstrates how to score genetic variants on their splicing
effect using AlphaGenome. This is the approach used in the AlphaGenome paper to
score ClinVar variants for missplicing, and is the recommended method to assess
whether a variant is causing aberrant splicing.

We use the "merged splicing" approach, which combines
three splicing-related variant scorers into a single score:

- **Splice sites**: Quantifies changes in splice site class assignment
  probabilities (donor, acceptor) between ALT and REF alleles.
- **Splice site usage**: Quantifies changes in the relative usage of splice
  sites between ALT and REF alleles.
- **Splice junctions**: Quantifies changes in the predicted splice junction
  usage between ALT and REF alleles.

The merged splicing score is computed as (described in the paper):

```
alphagenome_splicing = max(splice_sites) + max(splice_site_usage) + max(splice_junctions) / 5.0
```

This weighting scheme gives full weight to changes in splice site identity and
usage, but a smaller weight to junction-level changes because their magnitude is larger.

```{tip}
Open this tutorial in Google Colab for interactive viewing.
```

In [ ]:
# @title Install AlphaGenome

# @markdown Run this cell to install AlphaGenome.
from IPython.display import clear_output
! pip install alphagenome
clear_output()

## Imports

In [ ]:
from alphagenome import colab_utils
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import pandas as pd
from alphagenome.data import gene_annotation, genome, track_data, transcript
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load the model

In [ ]:
dna_model = dna_client.create(colab_utils.get_api_key('ag_key'))

In [ ]:
[output.name for output in dna_client.OutputType]

In [ ]:
# Load metadata objects for human.
output_metadata = dna_model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS
)

In [ ]:
# Load gene annotations (from GENCODE).
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg19/gencode.v19.annotation.gtf.gz.feather'
)
# Filter to protein-coding genes and highly supported transcripts.
gtf_transcript = gene_annotation.filter_transcript_support_level(
    gene_annotation.filter_protein_coding(gtf), ['1']
)

# Extractor for identifying transcripts in a region.
transcript_extractor = transcript.TranscriptExtractor(gtf_transcript)

# Also define an extractor that fetches only the longest transcript per gene.
gtf_longest_transcript = gene_annotation.filter_to_longest_transcript(
    gtf_transcript
)
longest_transcript_extractor = transcript.TranscriptExtractor(
    gtf_longest_transcript
)

# Gene expression
Gene expression is primarily captured by the model outputs RNA_SEQ and CAGE.Here is an example of expression predictions for colon tissue in a genomic interval containing the gene.Positions of the longest transcript per gene for genes in this interval are shown at the top:

In [ ]:
# Define interval to make predictions for (used throughout this tutorial).
# Note that the interval width must be one of the supported sequence lengths.
interval = genome.Interval('chr2', 227_085_120, 227_117_888).resize(
    dna_client.SEQUENCE_LENGTH_1MB
)

# Define the tissues/cell-types to predict expression for.
ontology_terms = [
    'UBERON:0002113',  # Colon - Sigmoid.
]

# Make predictions.
output = dna_model.predict_interval(
    interval=interval,
    requested_outputs={
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.CAGE,
    },
    ontology_terms=ontology_terms,
)

# Extract the longest transcripts per gene for this interval.
longest_transcripts = longest_transcript_extractor.extract(interval)

In [ ]:
# Build plot.
plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.Tracks(
            tdata=output.rna_seq,
            ylabel_template='RNA_SEQ: {biosample_name} ({strand})\n{name}',
        ),
        plot_components.Tracks(
            tdata=output.cage,
            ylabel_template='CAGE: {biosample_name} ({strand})\n{name}',
        ),
    ],
    interval=interval,
    title='Predicted RNA Expression (RNA_SEQ, CAGE) for colon tissue',
)

In [ ]:
[t.info['gene_name'] for t in longest_transcripts if t.strand == '+']

In [ ]:
# Define the variant of interest.
variant_string = 'chr2:227101504:G>T'
variant = genome.Variant.from_str(variant_string)
variant

In [ ]:
# Make predictions for sequences containing the REF and ALT alleles.
output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs={
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.CAGE,
    },
    ontology_terms=ontology_terms,
)

In [ ]:
# Zoom in on the region around COL4A4
col4a4_interval = gene_annotation.get_gene_interval(gtf, gene_symbol='COL4A4')

# Add 1KB on either side of the gene body.
col4a4_interval.resize_inplace(col4a4_interval.width + 1000)

In [ ]:
# Define the colors for REF and ALT predictions.
ref_alt_colors = {'REF': 'dimgrey', 'ALT': 'red'}

# Build plot.
plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),
        # RNA-seq tracks.
        plot_components.OverlaidTracks(
            tdata={
                'REF': output.reference.rna_seq.filter_to_nonpositive_strand(),
                'ALT': output.alternate.rna_seq.filter_to_nonpositive_strand(),
            },
            colors=ref_alt_colors,
            ylabel_template='{biosample_name} ({strand})\n{name}',
        ),
        # CAGE track.
        plot_components.OverlaidTracks(
            tdata={
                'REF': output.reference.cage.filter_to_nonpositive_strand(),
                'ALT': output.alternate.cage.filter_to_nonpositive_strand(),
            },
            colors=ref_alt_colors,
            ylabel_template='{biosample_name} ({strand})\n{name}',
        ),
    ],
    annotations=[plot_components.VariantAnnotation([variant])],
    interval=col4a4_interval,
    title='Effect of variant on predicted RNA Expression in colon tissue',
)

## Define the splicing variant scorers

AlphaGenome provides three variant scorers related to splicing:

- `GeneMaskSplicingScorer` with `OutputType.SPLICE_SITES`: Measures changes in
  splice site classification probabilities (donor, acceptor, neither).
- `GeneMaskSplicingScorer` with `OutputType.SPLICE_SITE_USAGE`: Measures changes
  in splice site usage levels.
- `SpliceJunctionScorer`: Measures changes in splice junction predictions.

We can access these from the recommended scorers dictionary:

In [ ]:
splicing_scorers = [
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITES'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_SITE_USAGE'],
    variant_scorers.RECOMMENDED_VARIANT_SCORERS['SPLICE_JUNCTIONS'],
]

for scorer in splicing_scorers:
  print(f'{scorer.name} (signed={scorer.is_signed})')

## Score a single variant

Let's score a variant near a known splice site. We use a 1MB sequence length, as
this provides the best predictions for gene-level scoring.

In [ ]:
# Define a variant near a splice site in the BRCA2 gene.
variant = genome.Variant(
    chromosome='chr2',
    position=227101504,
    reference_bases='G',
    alternate_bases='T',
)

# Create a 1MB interval centered on the variant.
interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

# Score the variant using the splicing scorers.
scores = dna_model.score_variant(
    interval=interval,
    variant=variant,
    variant_scorers=splicing_scorers,
    organism=dna_client.Organism.HOMO_SAPIENS,
)

# View the tidy scores.
df_scores = variant_scorers.tidy_scores([scores])
df_scores

## Compute the merged splicing score

The merged splicing score combines the three individual splicing scores. For each
scorer, we take the maximum absolute score across all genes and tracks, then
combine them with the following weights:

- **Splice sites**: weight = 1.0
- **Splice site usage**: weight = 1.0
- **Splice junctions**: weight = 0.2 (i.e. divided by 5)

This combination captures both the disruption of splice site identity and the
downstream effect on junction usage.

In [ ]:
def compute_merged_splicing_score(
    df: pd.DataFrame,
) -> pd.DataFrame:
  """Computes the merged splicing score from tidy variant scores.

  For each variant, takes the max raw_score across genes and tracks for each
  splicing scorer, then combines them as:
    merged_splicing = max(splice_sites) + max(splice_site_usage)
                      + max(splice_junctions) / 5

  Args:
    df: Tidy scores DataFrame from variant_scorers.tidy_scores().

  Returns:
    DataFrame with one row per variant and columns for each splicing scorer's
    max score plus the merged splicing score.
  """
  # Get the max score per variant per output type.
  df['variant_id'] = df['variant_id'].map(str)
  max_scores = (
      df.groupby(['variant_id', 'output_type'])['raw_score']
      .max()
      .reset_index()
      .pivot(index='variant_id', columns='output_type', values='raw_score')
      .fillna(0.0)
  )

  # Compute the merged splicing score with the appropriate weights.
  max_scores['alphagenome_splicing'] = (
      max_scores.get('SPLICE_SITES', 0.0)
      + max_scores.get('SPLICE_SITE_USAGE', 0.0)
      + max_scores.get('SPLICE_JUNCTIONS', 0.0) / 5.0
  )

  return max_scores.reset_index()


merged_scores = compute_merged_splicing_score(df_scores)
merged_scores

## Score a batch of variants

You can score multiple variants and compute the merged splicing score for each:

In [ ]:
# Define a batch of variants to score.
variants = [
    genome.Variant(
        chromosome='chr13',
        position=32316462,
        reference_bases='T',
        alternate_bases='G',
    ),
    genome.Variant(
        chromosome='chr17',
        position=43092919,
        reference_bases='G',
        alternate_bases='A',
    ),
    genome.Variant(
        chromosome='chr7',
        position=117559593,
        reference_bases='T',
        alternate_bases='A',
    ),
]

# Score each variant and collect results.
all_scores = []
for variant in variants:
  interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)
  scores = dna_model.score_variant(
      interval=interval,
      variant=variant,
      variant_scorers=splicing_scorers,
      organism=dna_client.Organism.HOMO_SAPIENS,
  )
  all_scores.append(scores)

# Tidy all scores into a single DataFrame.
df_all_scores = variant_scorers.tidy_scores(all_scores)

# Compute merged splicing scores for all variants.
merged_scores = compute_merged_splicing_score(df_all_scores)
merged_scores

## Score interpretation

The score is in theory unbounded from 0 to infinity. Because the splice junction scorer is computing log fold change. However, emperically the majority of variants are in the range `[0, 6]`.

We are working on a recommended variant score cutoff. We see in practice variants with score > 1.0 generally have large effect.

# Chromatin accessibility
Chromatin accessibility is primarily captured by the model outputs DNASE and ATAC. Here is an example of predictions for the same genomic interval containing the gene. There are a large variety of tissues and cell-types available, but let's focus on plotting predictions for intestinal tract tissues:

In [ ]:
output = dna_model.predict_interval(
    interval,
    requested_outputs={
        dna_client.OutputType.DNASE,
        dna_client.OutputType.ATAC,
    },
    ontology_terms=['UBERON:0002113']
)

In [ ]:
# Two putative promoter intervals from Ensembl.
promoter_intervals = [
    genome.Interval(
        'chr2', 227_085_120, 227_117_888, name='Ensembl_promoter:ENSR227101504'
    ),
]

In [ ]:
longest_transcript_extractor = transcript.TranscriptExtractor(
    gtf_longest_transcript
)

longest_transcripts = longest_transcript_extractor.extract(interval)

plot_components_list = [
    plot_components.TranscriptAnnotation(longest_transcripts),
]

if output.dnase is not None:
    plot_components_list.append(
        plot_components.Tracks(
            tdata=output.dnase,
            ylabel_template='DNASE: {biosample_name} ({strand})\n{name}',
        )
    )

if output.atac is not None:
    plot_components_list.append(
        plot_components.Tracks(
            tdata=output.atac,
            ylabel_template='ATAC: {biosample_name} ({strand})\n{name}',
        )
    )

plot = plot_components.plot(
    plot_components_list,
    # Plot an 8kb window around the variant.
    interval=variant.reference_interval.resize(24000),
    annotations=[
        plot_components.VariantAnnotation([variant]),
        plot_components.IntervalAnnotation(promoter_intervals),
    ],
    title='Predicted chromatin accessibility (DNASE, ATAC) for colon tissue',
)

# Splicing
Splicing effects are primarily captured by the model outputs SPLICE_SITES, SPLICE_SITE_USAGE, and SPLICE_JUNCTIONS. Here is an example of splicing predictions in a genomic interval containing the gene.

Since RNA_SEQ data also captures splicing patterns, we can plot it together with the splicing outputs, allowing us to see how predictions from different modalities are related:

In [ ]:
# Make predictions for splicing outputs and RNA_SEQ.
output = dna_model.predict_interval(
    interval=interval,
    requested_outputs={
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.SPLICE_SITES,
        dna_client.OutputType.SPLICE_SITE_USAGE,
        dna_client.OutputType.SPLICE_JUNCTIONS,
    },
    ontology_terms=['UBERON:0002113'],
)
output.splice_sites.metadata

In [ ]:
# Since is on the negative DNA strand, we use `filter_negative_strand` to
# consider only negative stranded splice predictions.
plot_components_list = [
    plot_components.TranscriptAnnotation(longest_transcripts),
]

if output.splice_sites is not None:
    plot_components_list.append(
        plot_components.Tracks(
            tdata=output.splice_sites.filter_to_negative_strand(),
            ylabel_template='SPLICE SITES: {name} ({strand})',
        )
    )

plot = plot_components.plot(
    plot_components_list,
    interval=interval,
    title='Predicted splicing effects for colon tissue',
)

In [ ]:
# Make predictions for REF and ALT alleles.
output = dna_model.predict_variant(
    interval=interval,
    variant=variant,
    requested_outputs={
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.SPLICE_SITES,
        dna_client.OutputType.SPLICE_SITE_USAGE,
        dna_client.OutputType.SPLICE_JUNCTIONS,
    },
    ontology_terms=ontology_terms,
)

In [ ]:
# Get all transcripts, not just the longest one per gene.
transcripts = transcript_extractor.extract(interval)

ref_output = output.reference
alt_output = output.alternate

# Build plot.
plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts),
        plot_components.Sashimi(
            ref_output.splice_junctions
            .filter_to_strand('-')
            .filter_by_tissue('Colon_Transverse'),
            ylabel_template='Reference {biosample_name} ({strand})\n{name}',
        ),
        plot_components.Sashimi(
            alt_output.splice_junctions
            .filter_to_strand('-')
            .filter_by_tissue('Colon_Transverse'),
            ylabel_template='Alternate {biosample_name} ({strand})\n{name}',
        ),
        plot_components.OverlaidTracks(
            tdata={
                'REF': ref_output.rna_seq.filter_to_nonpositive_strand(),
                'ALT': alt_output.rna_seq.filter_to_nonpositive_strand(),
            },
            colors=ref_alt_colors,
            ylabel_template='RNA_SEQ: {biosample_name} ({strand})\n{name}',
        ),
        plot_components.OverlaidTracks(
            tdata={
                'REF': ref_output.splice_sites.filter_to_nonpositive_strand(),
                'ALT': alt_output.splice_sites.filter_to_nonpositive_strand(),
            },
            colors=ref_alt_colors,
            ylabel_template='SPLICE SITES: {name} ({strand})',
        ),
        plot_components.OverlaidTracks(
            tdata={
                'REF': (
                    ref_output.splice_site_usage.filter_to_nonpositive_strand()
                ),
                'ALT': (
                    alt_output.splice_site_usage.filter_to_nonpositive_strand()
                ),
            },
            colors=ref_alt_colors,
            ylabel_template=(
                'SPLICE SITE USAGE: {biosample_name} ({strand})\n{name}'
            ),
        ),
    ],
    interval=interval,
    annotations=[plot_components.VariantAnnotation([variant])],
    title='Predicted REF vs. ALT effects of variant in colon tissue',
)

# ChIP-Histone
Histone modification marks are captured by the CHIP_HISTONE output. Here is an example of predictions in the same interval and tissues for histone marks thought to indicate key regulatory effects (e.g., H3K4me3).

Some additional considerations for visualizing CHIP_HISTONE predictions:

ChIP-Histone (and ChIP-TF) predictions are returned at a coarser resolution (128bp), so we need to adjust the interval size plotted to be compatible (i.e., its width must be a multiple of 128). This is done automatically by the plotting library.
Not all biosamples have predictions for all histone marks, but four of the major ones (H3K4me3, H3K4me1, H3K27ac, H3K27me3 and H3K36me3) are available for 40% of biosamples. Let's see which histone marks are covered by biosamples corresponding to colon tissue:

In [ ]:
output_metadata.chip_histone[
    output_metadata.chip_histone['biosample_name'].str.contains('colon')
]

In [ ]:
# Make predictions.
output = dna_model.predict_interval(
    interval=interval,
    requested_outputs={dna_client.OutputType.CHIP_HISTONE},
    ontology_terms=['UBERON:0002113'],
)

In [ ]:
gtf_tss = gene_annotation.extract_tss(gtf_longest_transcript)

tss_as_intervals = [
    genome.Interval(
        chromosome=row.Chromosome,
        start=row.Start,
        end=row.End + 1000,  # Add extra 1Kb so the TSSs are visible.
        name=row.gene_name,
    )
    for _, row in gtf_tss.iterrows()
]

In [ ]:
reordered_chip_histone = output.chip_histone.select_tracks_by_index(
    output.chip_histone.metadata.sort_values('histone_mark').index
)

histone_to_color = {
    'H3K27AC': '#e41a1c',
    'H3K36ME3': '#ff7f00',
    'H3K4ME1': '#377eb8',
    'H3K4ME3': '#984ea3',
    'H3K9AC': '#4daf4a',
    'H3K27ME3': '#ffc0cb',
}

track_colors = (
    reordered_chip_histone.metadata['histone_mark']
    .map(lambda x: histone_to_color.get(x.upper(), '#000000'))
    .values
)

In [ ]:
# Build plot.
plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.Tracks(
            tdata=reordered_chip_histone,
            ylabel_template=(
                'CHIP HISTONE: {biosample_name} ({strand})\n{histone_mark}'
            ),
            filled=True,
            track_colors=track_colors,
        ),
    ],
    interval=interval,
    annotations=[
        plot_components.IntervalAnnotation(
            tss_as_intervals, alpha=0.5, colors='blue'
        )
    ],
    despine_keep_bottom=True,
    title='Predicted histone modification markers in colon tissue',
)

# ChIP-TF
Transcription factor (TF) binding patterns are captured by the model's CHIP_TF output. We will visualize these in a similar way to CHIP_HISTONE predictions, with one difference: we need to modify the ontology terms we request in order to get good coverage of both biosamples of interest and multiple TFs. Many biosamples have predictions for two major TFs, namely CTCF and POLR2A, but some cell-lines (HepG2 and K562) have coverage over many more TFs (501 and 269 respectively).

Here, we will also demonstrate two helpful utilities: selecting specific tracks from a TrackData object and aggregating predictions across tracks in a TrackData object.

In [ ]:
output = dna_model.predict_interval(
    interval=interval,
    requested_outputs={dna_client.OutputType.CHIP_TF},
    ontology_terms='UBERON:0002113',
)

In [ ]:
output_chip_tf = output.chip_tf.filter_tracks(
    (output.chip_tf.metadata['ontology_curie'].isin('UBERON:0002113')).values
)
len(output_chip_tf.metadata)

In [ ]:
max_predictions = output_chip_tf.metadata[
    ['ontology_curie', 'biosample_name', 'transcription_factor']
].copy()

max_predictions.loc[:, 'max_prediction'] = output_chip_tf.values.max(axis=0)
max_predictions.sort_values('max_prediction', ascending=False).reset_index(
    drop=True
)

In [ ]:
# Build plot.
plot_components.plot(
    components=[
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.Tracks(
            tdata=output_filtered,
            ylabel_template=(
                'CHIP TF: {biosample_name} ({strand})\n{transcription_factor}'
            ),
            filled=True,
        ),
    ],
    interval=interval,
    title='Predicted TF-binding in K562 and HepG2 cell-lines.',
    despine_keep_bottom=True,
    annotations=[
        plot_components.IntervalAnnotation(
            tss_as_intervals, alpha=0.3, colors='blue'
        )
    ],
)
plt.show()

In [ ]:
# Compute max predicted values per track in the APOL4 gene interval.
max_predictions = output.chip_tf.slice_by_interval(
    col4a4_interval, match_resolution=True
).values.max(axis=0)

# Filter to the 10 tracks with the highest predictions.
output_filtered = output.chip_tf.filter_tracks(
    (max_predictions >= np.sort(max_predictions)[-10])
)

# Build plot.
plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts),
        plot_components.Tracks(
            tdata=output_filtered,
            ylabel_template=(
                'CHIP TF: {biosample_name} ({strand})\n{transcription_factor}'
            ),
            filled=True,
        ),
    ],
    interval=apol4_interval,
    annotations=[plot_components.IntervalAnnotation(promoter_intervals)],
    despine_keep_bottom=True,
    title='Predicted TF-binding in K562, HepG2, and sigmoid colon.',
)
plt.show()

In [ ]:
mean_ctcf = output_filtered.values[
    :, output_filtered.metadata['transcription_factor'] == 'CTCF'
].mean(axis=1)

# Construct a new TrackData object from the mean values.
tdata_mean_ctcf = track_data.TrackData(
    values=mean_ctcf[:, None],
    metadata=pd.DataFrame(
        {'transcription_factor': ['CTCF'], 'name': ['mean'], 'strand': ['.']}
    ),
    interval=output_filtered.interval,
    resolution=output_filtered.resolution,
)

In [ ]:
plot_components.plot(
    [
        plot_components.TranscriptAnnotation(transcripts),
        plot_components.Tracks(
            tdata=tdata_mean_ctcf,
            ylabel_template='{name} {transcription_factor}',
            filled=True,
        ),
    ],
    interval=apol4_interval,
    annotations=[plot_components.IntervalAnnotation(promoter_intervals)],
    despine_keep_bottom=True,
    title='Predicted CTCF binding (mean across cell types)',
)
plt.show()

# Contact maps
Relative frequency of physical contacts between pairwise genetic positions are captured by the model's CONTACT_MAPS output. This output type is provided at even coarser resolution (2048bp), so we can only consider intervals that span distances larger than this.

Let's make the contact map predictions for a colon cell line:

In [ ]:
output = dna_model.predict_interval(
    interval=interval,
    requested_outputs={dna_client.OutputType.CONTACT_MAPS},
    ontology_terms='UBERON:0002113',
)

In [ ]:
plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.ContactMaps(
            tdata=output.contact_maps,
            ylabel_template='{biosample_name}\n{name}',
            cmap='autumn_r',
            vmax=1.0,
        ),
    ],
    interval=interval,
    title='Predicted contact maps',
)
plt.show()